# VioSpice ML Training Notebook

This notebook shows how to load VioSpice JSONL output, inspect records, build PyTorch datasets, and train a small regression model.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
examples_dir = repo_root / "examples" / "ml_api"
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))

from pytorch_dataset import load_jsonl, VioSpiceJsonlDataset

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split

In [ ]:
DATASET_PATH = Path('/tmp/viospice-datasets/amp.jsonl')
LABEL_NAMES = ['gain_ratio']
STAT_FEATURES = [('ALL', 'avg'), ('ALL', 'max'), ('ALL', 'rms')]
PARAM_FEATURES = ['vin_dc']
SEED = 42
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

assert DATASET_PATH.exists(), f'Dataset not found: {DATASET_PATH}'

In [ ]:
records = load_jsonl(str(DATASET_PATH), accepted_only=True)
print('records:', len(records))
records[0].keys() if records else None

In [ ]:
if records:
    print('labels:', records[0]['labels'])
    print('metadata:', records[0]['metadata'])
    print('stats:', records[0]['artifacts']['stats'][:2])

In [ ]:
dataset = VioSpiceJsonlDataset(
    str(DATASET_PATH),
    label_names=LABEL_NAMES,
    stat_features=STAT_FEATURES,
    param_features=PARAM_FEATURES,
    accepted_only=True,
)
sample_x, sample_y = dataset[0]
print('feature_dim:', sample_x.shape[0])
print('label_dim:', sample_y.shape[0])
print('sample_x:', sample_x)
print('sample_y:', sample_y)

In [ ]:
generator = torch.Generator().manual_seed(SEED)
train_size = max(1, int(len(dataset) * 0.8))
val_size = max(1, len(dataset) - train_size)
if train_size + val_size > len(dataset):
    train_size = len(dataset) - 1
    val_size = 1

train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=generator)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
len(train_ds), len(val_ds)

In [ ]:
class Regressor(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Regressor(sample_x.shape[0], sample_y.shape[0]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

In [ ]:
def evaluate(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            pred = model(x)
            loss = loss_fn(pred, y)
            losses.append(float(loss.item()))
    return sum(losses) / max(1, len(losses))

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.item()))
    train_loss = sum(epoch_losses) / max(1, len(epoch_losses))
    val_loss = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss))
    print(f'epoch={epoch:02d} train_loss={train_loss:.6f} val_loss={val_loss:.6f}')

In [ ]:
with torch.no_grad():
    pred = model(sample_x.unsqueeze(0).to(device)).cpu().squeeze(0)
print('sample target:', sample_y.tolist())
print('sample prediction:', pred.tolist())